# SageMaker Studio で Amazon SageMaker パイプラインと SageMaker モデルレジストリを使用する

このラボでは、Amazon Sagemaker Pipeline を作成して実行し、パイプラインの進捗状況をモニタリングします。また、機械学習 (ML) プロセスが使用または生成するアーティファクトの一部を見つけて調べることもできます。

時間が許せば、パイプラインが生成したモデルの系統の詳細を確認することもできます。

### タスク 2.1: 環境のセットアップ

SageMaker Pipeline を作成する前に、必要なパッケージをインストールし、モジュールをインポートし、サポートファイルをステージングして環境を準備する必要があります。この Sagemaker パイプラインは機能グループを使用するように設計されているため、Amazon SageMaker Feature Store で機能グループを作成し、Data Wrangler フローを実行して環境を準備することもできます。 

このタスクのセルを実行して次の操作を行います。
- 依存関係をインストールします。
- 必要なモジュールをインポートします。
- データとコードを Amazon Simple Storage Service (Amazon S3) にコピーしてください。
- 機能グループを作成します。
- フィーチャーをフィーチャーグループにインジェストします。

### タスク 2.2: 依存関係のインストール

In [ ]:
%%capture
#install dependencies
%pip install --upgrade pip 
%pip install pytest-astropy ==  0.7.0
%pip install rsa == 4.7.2
%pip install PyYAML
!apt update && apt install -y git
%pip install git+https://github.com/aws-samples/ml-lineage-helper

### タスク 2.3: モジュールのインポート

In [ ]:
#import-modules
import os
import json
import boto3
import sagemaker
import sagemaker.session
import datetime as dt
import pandas as pd
import time
from time import gmtime, strftime
import uuid
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
from sagemaker.model_metrics import (
    MetricsSource,
    ModelMetrics,
)
from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
    ScriptProcessor,
)
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.workflow.conditions import ConditionGreaterThan
from sagemaker.workflow.parameters import (
    ParameterInteger,
    ParameterString,
)
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import (
    ProcessingStep,
    TrainingStep,
)
from sagemaker.workflow.condition_step import (
    ConditionStep,
    JsonGet,
)
from sagemaker.workflow.step_collections import RegisterModel
from sagemaker.model import Model
from sagemaker.workflow.steps import CreateModelStep
from sagemaker.inputs import CreateModelInput
from sagemaker.inputs import TransformInput
from sagemaker.workflow.steps import TransformStep
from sagemaker.transformer import Transformer
from sagemaker.pytorch.estimator import PyTorch
from sagemaker.tuner import HyperparameterTuner
from sagemaker.inputs import TrainingInput
from sagemaker.workflow.steps import TuningStep
from sagemaker.tuner import (
    IntegerParameter,
    CategoricalParameter,
    ContinuousParameter,
    HyperparameterTuner,
)
from ml_lineage_helper import *
from sagemaker.feature_store.feature_definition import FeatureDefinition
from sagemaker.feature_store.feature_definition import FeatureTypeEnum
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.processing import FeatureStoreOutput
from sagemaker.processing import Processor
from sagemaker.network import NetworkConfig
from sagemaker.dataset_definition.inputs import AthenaDatasetDefinition, DatasetDefinition, RedshiftDatasetDefinition

In [ ]:
#create sessions
boto_session  =  boto3.Session()
sagemaker_session = sagemaker.Session()


In [ ]:
#create clients
s3_client = boto3.client('s3')
featurestore_runtime = boto3.client('sagemaker-featurestore-runtime')
sagemaker_client = boto3.client('sagemaker')

In [ ]:
#feature store session
feature_store_session = Session(
    boto_session = boto_session,
    sagemaker_client = sagemaker_client,
    sagemaker_featurestore_runtime_client = featurestore_runtime
)

In [ ]:
#set global variables
default_bucket = sagemaker_session.default_bucket()
region = boto_session.region_name
role = sagemaker.get_execution_role()

### タスク 2.4: ラボファイルを Amazon S3 にコピーする

In [ ]:
# Upload files to default bucket
s3_client.put_object(Bucket = default_bucket, Key = 'data/')
s3_client.put_object(Bucket = default_bucket, Key = 'input/code/')
s3_client.upload_file('pipelines/data/storedata_total.csv', default_bucket, 'data/storedata_total.csv')
s3_client.upload_file('pipelines/input/code/evaluate.py', default_bucket, 'input/code/evaluate.py')
s3_client.upload_file('pipelines/input/code/generate_config.py', default_bucket, 'input/code/generate_config.py')
s3_client.upload_file('pipelines/input/code/processfeaturestore.py', default_bucket, 'input/code/processfeaturestore.py')

# Preview the dataset
print('Dataset preview:')
customer_data = pd.read_csv('pipelines/data/storedata_total.csv')
customer_data.head()

### タスク 2.5: 機能グループの作成

このタスクでは、データの機能グループを作成します。まず、データのスキーマを作成します。このラボでは、スキーマを最初に **name** 列でソートし、次に **type** 列でソートする必要があります。

In [ ]:
#set-up-feature-store-variables
record_identifier_feature_name = 'FS_ID'
event_time_feature_name = 'FS_time'

column_schemas = [
    {
        "name": "retained",
        "type": "long"
    },
    {
        "name": "esent",
        "type": "long"
    },
    {
        "name": "eopenrate",
        "type": "float"
    },
    {
        "name": "eclickrate",
        "type": "float"
    },
    {
        "name": "avgorder",
        "type": "float"
    },
    {
        "name": "ordfreq",
        "type": "float"
    },
    {
        "name": "paperless",
        "type": "long"
    },
    {
        "name": "refill",
        "type": "long"
    },
    {
        "name": "doorstep",
        "type": "long"
    },
    {
        "name": "first_last_days_diff",
        "type": "long"
    },
    {
        "name": "created_first_days_diff",
        "type": "long"
    },
    {
        "name": "favday_Friday",
        "type": "long"
    },
    {
        "name": "favday_Monday",
        "type": "long"
    },
    {
        "name": "favday_Saturday",
        "type": "long"
    },
    {
        "name": "favday_Sunday",
        "type": "long"
    },
    {
        "name": "favday_Thursday",
        "type": "long"
    },
    {
        "name": "favday_Tuesday",
        "type": "long"
    },
    {
        "name": "favday_Wednesday",
        "type": "long"
    },
    {
        "name": "city_BLR",
        "type": "long"
    },
    {
        "name": "city_BOM",
        "type": "long"
    },
    {
        "name": "city_DEL",
        "type": "long"
    },
    {
        "name": "city_MAA",
        "type": "long"
    },
    {
        "name": "FS_ID",
        "type": "long"
    },
    {
        "name": "FS_time",
        "type": "float"
    }
]


次に、機能グループを作成します。

In [ ]:
# Flow name and a unique ID for this export (used later as the processing job name for the export)
flow_name = 'featureengineer'
flow_export_id = f"{strftime('%d-%H-%M-%S', gmtime())}-{str(uuid.uuid4())[:8]}"
flow_export_name = f"flow-{flow_export_id}"

# Feature group name, with flow_name and a unique id. You can give it a customized name
feature_group_name = f"FG-{flow_name}-{str(uuid.uuid4())[:8]}"

# SageMaker Feature Store writes the data in the offline store of a Feature Group to a 
# Amazon S3 location owned by you.
feature_store_offline_s3_uri = 's3://' + default_bucket

# Controls if online store is enabled. Enabling the online store allows quick access to 
# the latest value for a record by using the GetRecord API.
enable_online_store = True

In [ ]:
#create-feature-group
default_feature_type = FeatureTypeEnum.STRING
column_to_feature_type_mapping = {
    "float": FeatureTypeEnum.FRACTIONAL,
    "long": FeatureTypeEnum.INTEGRAL
}

feature_definitions = [
    FeatureDefinition(
        feature_name = column_schema['name'], 
        feature_type = column_to_feature_type_mapping.get(column_schema['type'], default_feature_type)
    ) for column_schema in column_schemas
]


print(f"Feature Group Name: {feature_group_name}")

# Confirm the Athena settings are configured
try:
    boto3.client('athena').update_work_group(
        WorkGroup = 'primary',
        ConfigurationUpdates = {
            'EnforceWorkGroupConfiguration':False
        }
    )
except Exception:
    pass

feature_group = FeatureGroup(
    name = feature_group_name, sagemaker_session = feature_store_session, feature_definitions = feature_definitions)

feature_group.create(
    s3_uri = feature_store_offline_s3_uri,
    record_identifier_name = record_identifier_feature_name,
    event_time_feature_name = event_time_feature_name,
    role_arn = role,
    enable_online_store = enable_online_store
)

def wait_for_feature_group_creation_complete(feature_group):
    """Helper function to wait for the completions of creating a feature group"""
    response = feature_group.describe()
    status = response.get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for feature group creation")
        time.sleep(5)
        response = feature_group.describe()
        status = response.get("FeatureGroupStatus")

    if status != "Created":
        print(f"Failed to create feature group, response: {response}")
        failureReason = response.get("FailureReason", "")
        raise SystemExit(
            f"Failed to create feature group {feature_group.name}, status: {status}, reason: {failureReason}"
        )
    print(f"Feature Group {feature_group.name} successfully created.")

wait_for_feature_group_creation_complete(feature_group = feature_group)


### タスク 2.6: フィーチャの取り込み

このプロセスが完了するまでに約 8 分かかります。

In [ ]:
#populate-feature-store
column_list = ['retained','esent','eopenrate','eclickrate','avgorder','ordfreq','paperless','refill','doorstep','first_last_days_diff','created_first_days_diff','favday_Friday','favday_Monday', 'favday_Saturday','favday_Sunday','favday_Thursday','favday_Tuesday','favday_Wednesday','city_BLR','city_BOM','city_DEL','city_MAA','FS_ID','FS_time']
lab_test_data = pd.read_csv('featureengineer_data/store_data_processed.csv', names = (column_list), header = 1)
feature_group.ingest(data_frame = lab_test_data, wait = True)

### タスク 2.7: SageMaker パイプラインを作成して実行する

環境が整ったので、SageMaker Pipeline を設定、作成、起動します。

SageMaker Pipeline は、一連の依存ステップを実行するワークフローです。ステップは入力を受け付けて出力を送信できるため、データやその他のアセットをステップ間で受け渡すことができます。

次のセルでコードを実行して次の操作を行います。
- パイプラインの設定に必要な変数を定義する
- SageMaker セッションを設定する
- パイプラインステップを定義する
- パイプラインを設定する
- パイプラインを作成する
- パイプラインを起動する
- パイプラインについて説明する
- 待機イベントを作成して、パイプラインの実行が終了するまでノートブックが処理を続行しないようする

### タスク 2.8: パイプラインが使用する変数を設定する

In [ ]:

#pipeline-variables
feature_group_name = feature_group.name
model_name = "Churn-model"

sklearn_processor_version = "0.23-1"
model_package_group_name = "ChurnModelPackageGroup"
pipeline_name = "ChurnModelSMPipeline"

processing_instance_count = ParameterInteger(
    name = "ProcessingInstanceCount",
    default_value = 1
    )

processing_instance_type = ParameterString(
        name = "ProcessingInstanceType",
        default_value = "ml.m5.xlarge"
    )

training_instance_type = ParameterString(
        name = "TrainingInstanceType",
        default_value = "ml.m5.xlarge"
    )

input_data = ParameterString(
        name = "InputData",
        default_value = "s3://{}/data/storedata_total.csv".format(default_bucket), 
    )

batch_data = ParameterString(
        name = "BatchData",
        default_value = "s3://{}/data/batch/batch.csv".format(default_bucket),
    )

### タスク 2.9: パイプラインの設定

**ChurnModelPipeline** という名前のパイプラインを定義して、顧客を維持または失う可能性を評価するモデルを作成します。このパイプラインには 9 つのステップがあります。

パイプラインの各ステップは特定のジョブタイプを実行します。ジョブに必要な入力は、ジョブのタイプによって異なります。SageMaker パイプラインステップタイプの詳細については、[Step Types](https://docs.aws.amazon.com/sagemaker/latest/dg/build-and-manage-steps.html#build-and-manage-steps-types) を参照してください。

次のセルのコードを確認して、各ステップがどのように定義されているかを理解してください。

**ChurnModelProcess** ステップは **step_process** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Processing – Processing ジョブは ProcessingStep() クラスを使用して定義されます。
- **Processor:** SKLearnProcessor.
- **Destination:** 出力はデフォルトの S3 バケットの下のフォルダに送信されます。
- **Job Arguments:** このステップでは、フィーチャストアを使用してデータセットを処理します。
- **Code:** **processfeaturestore.py** これはデフォルトの S3 バケットにあります。


In [ ]:
#configure-processing-step
# Run a scikit-learn script to do data processing on SageMaker 
# using the SKLearnProcessor class
sklearn_processor = SKLearnProcessor(
        framework_version = sklearn_processor_version,
        instance_type = processing_instance_type.default_value, 
        instance_count = processing_instance_count,
        sagemaker_session = sagemaker_session,
        role = role,
    )

# Inputs, outputs, and code are parameters to the processor
# step_* will become the pipeline steps toward the end of the cell
# in this case, use the feature store as input, so there is no externalinput
step_process = ProcessingStep(
        name = "ChurnModelProcess",
        processor = sklearn_processor,
        outputs = [
            ProcessingOutput(output_name = "train", source = "/opt/ml/processing/train",\
                             destination = f"s3://{default_bucket}/output/train" ),
            ProcessingOutput(output_name = "validation", source = "/opt/ml/processing/validation",\
                            destination = f"s3://{default_bucket}/output/validation"),
            ProcessingOutput(output_name = "test", source = "/opt/ml/processing/test",\
                            destination = f"s3://{default_bucket}/output/test"),
            ProcessingOutput(output_name = "batch", source = "/opt/ml/processing/batch",\
                            destination = f"s3://{default_bucket}/data/batch"),
            ProcessingOutput(output_name = "baseline", source = "/opt/ml/processing/baseline",\
                            destination = f"s3://{default_bucket}/input/baseline")
        ],
        job_arguments = ["--featuregroupname",feature_group_name,"--default-bucket",default_bucket,"--region",region],
        code = f"s3://{default_bucket}/input/code/processfeaturestore.py",
    )

**ChurnHyperParameterTuning** ステップは **step_tuning** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Tuning – Tuning ジョブは TuningStep() クラスを使用して定義されます。
- **Tuner:** このジョブは XGBoost フレームワークを使用しています。
- **Inputs:** このジョブは **step_process** という名前の ChurnModelProcess ステップによって生成されたトレーニングおよび検証データを使用していることに注意してください。

In [ ]:
#configure-churn-hyperparameter-tuning
# Training/tuning step for generating model artifacts
model_path = f"s3://{default_bucket}/output"
image_uri = sagemaker.image_uris.retrieve(
    framework = "xgboost",
    region = region,
    version = "1.5-1",
    py_version = "py3",
    instance_type = training_instance_type.default_value,
)

fixed_hyperparameters = {
    "eval_metric":"auc",
    "objective":"binary:logistic",
    "num_round":"100",
    "rate_drop":"0.3",
    "tweedie_variance_power":"1.4"
    }

xgb_train = Estimator(
    image_uri = image_uri,
    instance_type = training_instance_type,
    instance_count = 1,
    hyperparameters = fixed_hyperparameters,
    output_path = model_path,
    base_job_name = f"churn-train",
    sagemaker_session = sagemaker_session,
    role = role
    )

In [ ]:
#Tuning steps
hyperparameter_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
    }
objective_metric_name = "validation:auc"

step_tuning = TuningStep(
    name = "ChurnHyperParameterTuning",
    tuner = HyperparameterTuner(xgb_train, objective_metric_name, hyperparameter_ranges, max_jobs = 2, max_parallel_jobs = 2),
    inputs = {
            "train": TrainingInput(
                s3_data = step_process.properties.ProcessingOutputConfig.Outputs[
                    "train"
                ].S3Output.S3Uri,
                content_type = "text/csv",
            ),
            "validation": TrainingInput(
                s3_data = step_process.properties.ProcessingOutputConfig.Outputs[
                    "validation"
                ].S3Output.S3Uri,
                content_type = "text/csv",
            ),
        },
    )

**ChurnEvalBestModel** ステップは **step_eval** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** 処理中
- **Processor:** ScriptProcessor
- **Inputs:** このジョブでは ChurnHyperParameterTuning (**step_tuning**) の最上位モデルと ChurnModelProcess (**step_process**) のテスト出力が使用されていることに注意してください。
- **Outputs:** 出力はデフォルトの S3 バケットに書き込まれます。
- **Code:** 評価には Amazon S3 にある **evaluate.py** という名前のスクリプトが使用されます。

In [ ]:
#configure-churn-best-model
evaluation_report = PropertyFile(
    name = "ChurnEvaluationReport",
    output_name = "evaluation",
    path = "evaluation.json",
)

script_eval = ScriptProcessor(
    image_uri = image_uri,
    command = ["python3"],
    instance_type = processing_instance_type,
    instance_count = 1,
    base_job_name = "script-churn-eval",
    role = role,
    sagemaker_session = sagemaker_session,
)

step_eval = ProcessingStep(
    name = "ChurnEvalBestModel",
    processor = script_eval,
    inputs = [
        ProcessingInput(
            source = step_tuning.get_top_model_s3_uri(top_k = 0, s3_bucket = default_bucket, prefix = "output"),
            destination = "/opt/ml/processing/model"
        ),
        ProcessingInput(
            source = step_process.properties.ProcessingOutputConfig.Outputs[
                "test"
            ].S3Output.S3Uri,
            destination = "/opt/ml/processing/test"
        )
    ],
    outputs = [
        ProcessingOutput(output_name = "evaluation", source = "/opt/ml/processing/evaluation",\
                            destination = f"s3://{default_bucket}/output/evaluation"),
    ],
    code = f"s3://{default_bucket}/input/code/evaluate.py",
    property_files = [evaluation_report],
)

**ChurnCreateModel** ステップは **step_create_model** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Model – モデルジョブは Model() クラスを使用して定義されます。
- **Model:** このステップで使用されるモデルは、以前に定義された **model** という名前の変数で定義されています。**model** 変数は ChurnHyperParameterTuning (**step_tuning**) によって作成された最上位モデルを使用していることに注意してください。
- **Inputs:** 入力にはインスタンスタイプとアクセラレータタイプが含まれます。

In [ ]:
#configure-model-creation
model = Model(
    image_uri = image_uri,        
    model_data = step_tuning.get_top_model_s3_uri(top_k = 0,s3_bucket = default_bucket,prefix = "output"),
    name = model_name,
    sagemaker_session = sagemaker_session,
    role = role,
)

inputs = CreateModelInput(
    instance_type = "ml.m5.large",
    accelerator_type = "ml.inf1.xlarge",
)

step_create_model = CreateModelStep(
    name = "ChurnCreateModel",
    model = model,
    inputs = inputs,
)

**ChurnModelConfigFile** ステップは **step_config_file** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** 処理中
- **Processor:** ScriptProcessor.
- **Code:** **generate_config.py**。これはデフォルトの S3 バケットにあります。
- **Job Arguments:** ジョブ引数には、**ChurnCreateModel** によって生成されたモデル、バイアスレポートへのパス、デフォルトバケット、サンプル数、および処理に使用されたインスタンス数が含まれます。
- **Depends On:** このジョブはモデルの作成が完了するまで実行できないことに注意してください。

In [ ]:

#configure-script-processing
bias_report_output_path = f"s3://{default_bucket}/clarify-output/bias"
clarify_instance_type = 'ml.m5.xlarge'
analysis_config_path = f"s3://{default_bucket}/clarify-output/bias/analysis_config.json"
clarify_image = sagemaker.image_uris.retrieve(framework = 'sklearn', version = sklearn_processor_version, region = region)

#custom_image_uri = None
script_processor = ScriptProcessor(
    command = ['python3'],
    image_uri = clarify_image,
    role = role,
    instance_count = 1,
    instance_type = processing_instance_type,
    sagemaker_session = sagemaker_session,
)

step_config_file = ProcessingStep(
    name = "ChurnModelConfigFile",
    processor = script_processor,
    code = f"s3://{default_bucket}/input/code/generate_config.py",
    job_arguments = ["--modelname", step_create_model.properties.ModelName, "--bias-report-output-path", bias_report_output_path, "--clarify-instance-type", clarify_instance_type,\
                  "--default-bucket", default_bucket, "--num-baseline-samples", "50", "--instance-count", "1"],
    depends_on = [step_create_model.name]
)

**ChurnTransform** ステップは **step_transform** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Transform – トランスフォームジョブは transformStep() クラスを使用して定義されます。
- **Transformer:** トランスフォーマーの詳細は、以前に定義された **transformer** という名前の変数に設定されます。この変数は ChurnCreateModel (**step_create_model**) で作成されたモデルを使用していることに注意してください。
- **Inputs:** 変換されるデータ、batch.csv。これはノートブックで以前に定義されていました。入力には、ファイルタイプと分割方法も含まれます。

In [ ]:
#configure-inference
transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type = "ml.m5.xlarge",
    instance_count = 1,
    assemble_with = "Line",
    accept = "text/csv",    
    output_path = f"s3://{default_bucket}/ChurnTransform"
    )

step_transform = TransformStep(
    name = "ChurnTransform",
    transformer = transformer,
    inputs = TransformInput(data = batch_data, content_type = "text/csv", join_source = "Input", split_type = "Line")
    )

**ClarifyProcessingStep** ステップは **step_clarify** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** 処理中
- **Processor:** このジョブは SageMaker ClarifyProcessor を使用しています。プロセッサ構成は、**clarify_processor** という名前の変数で確認できます。
- **Inputs:** 入力は **data_input** 変数と **congif_input** 変数で定義されています。
- **Outputs:** 出力はデフォルトバケットの下のフォルダに書き込まれます。
- **Depends On:**  Amazon SageMaker Clarify に必要な設定ファイルが **ChurnModelConfigFile** によって作成されるまで、このジョブは実行できないことに注意してください。

In [ ]:
#configure-clarify-processing
data_config = sagemaker.clarify.DataConfig(
s3_data_input_path = f's3://{default_bucket}/output/train/train.csv',
s3_output_path = bias_report_output_path,
    label = 0,
    headers = ['target','esent','eopenrate','eclickrate','avgorder','ordfreq','paperless','refill','doorstep','first_last_days_diff','created_first_days_diff','favday_Friday','favday_Monday','favday_Saturday','favday_Sunday','favday_Thursday','favday_Tuesday','favday_Wednesday','city_BLR','city_BOM','city_DEL','city_MAA'],
    dataset_type = "text/csv",
)

clarify_processor = sagemaker.clarify.SageMakerClarifyProcessor(
    role = role,
    instance_count = 1,
    instance_type = clarify_instance_type,
    sagemaker_session = sagemaker_session,
)

config_input = ProcessingInput(
    input_name = "analysis_config",
    source=analysis_config_path,
    destination = "/opt/ml/processing/input/analysis_config",
    s3_data_type = "S3Prefix",
    s3_input_mode = "File",
    s3_compression_type = "None",
    )

data_input = ProcessingInput(
    input_name = "dataset",
    source = data_config.s3_data_input_path,
    destination = "/opt/ml/processing/input/data",
    s3_data_type = "S3Prefix",
    s3_input_mode = "File",
    s3_data_distribution_type = data_config.s3_data_distribution_type,
    s3_compression_type = data_config.s3_compression_type,
)

result_output = ProcessingOutput( 
    source = "/opt/ml/processing/output",
    destination = data_config.s3_output_path,
    output_name = "analysis_result",
    s3_upload_mode = "EndOfJob",
)

step_clarify = ProcessingStep(
    name = "ClarifyProcessingStep",
    processor = clarify_processor,
    inputs = [data_input, config_input],
    outputs = [result_output],
    depends_on = [step_config_file.name]
)

**RegisterChurnModel** ステップは **step_register** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Register Model – 登録ジョブは RegisterMode() クラスを使用して定義されます。
- **Estimator:** 推定器はセルの前半の **xgbtrain** 変数で定義されています。
- **Model Data:** これは **ChurnHyperParameterTuning** によって返されるモデル URI です。
- **Content Types:** text/csv
- **Response Types** text/csv
- **Inference Instance:** これは推論処理に使用されるインスタンスタイプです。
- **Transform Instance:** これは変換の処理に使用されるインスタンスタイプです。
- **Model Package Group Name:** これはモデルバージョンのグループを保存するグループの名前です。
-**Model Metrics:** これはモデルメトリクスの場所を定義します。含まれているファイルは、SageMaker Clarify バイアスレポート、SageMaker Clarify 説明可能性レポート、およびモデル評価です。

In [ ]:
#configure-model-registry
model_statistics = MetricsSource(
    s3_uri = "s3://{}/output/evaluation/evaluation.json".format(default_bucket),
    content_type = "application/json"
    )
explainability = MetricsSource(
    s3_uri = "s3://{}/clarify-output/bias/analysis.json".format(default_bucket),
    content_type = "application/json"
    )

bias = MetricsSource(
    s3_uri = "s3://{}/clarify-output/bias/analysis.json".format(default_bucket),
    content_type = "application/json"
    ) 

model_metrics = ModelMetrics(
    model_statistics = model_statistics,
    explainability = explainability,
    bias = bias
)

step_register = RegisterModel(
    name = "RegisterChurnModel",
    estimator = xgb_train,
    model_data = step_tuning.get_top_model_s3_uri(top_k = 0, s3_bucket = default_bucket, prefix = "output"),
    content_types = ["text/csv"],
    response_types = ["text/csv"],
    inference_instances = ["ml.t2.medium", "ml.m5.large"],
    transform_instances = ["ml.m5.large"],
    model_package_group_name = model_package_group_name,
    model_metrics = model_metrics,
)

**CheckAUCScoreChurnEvaluation** ステップは **step_cond** という名前の変数で定義されています。

ステップ構成には以下が含まれます。
- **Type:** Condition – 条件ジョブは ConditionStep() クラスを使用して定義されます。
- **Conditions:** **ChurnEvalBestModel** からの出力が 0.75 より大きい場合、この条件は True と評価されます。
- **If Steps:** これは、条件が True と評価された場合に実行されるステップのリストです。
- **Else Steps:** これは、条件が False と評価された場合に実行されるステップのリストです。このリストは空であることに注意してください。つまり、条件が満たされない場合、パイプラインは処理を停止します。

In [ ]:
%%capture
cond_lte = ConditionGreaterThan(
    left = JsonGet(
        step = step_eval,
        property_file = evaluation_report,
        json_path = "binary_classification_metrics.auc.value"
    ),
    right = 0.75,
)

step_cond = ConditionStep(
    name = "CheckAUCScoreChurnEvaluation",
    conditions = [cond_lte],
    if_steps = [step_create_model, step_config_file, step_transform, step_clarify, step_register],
    else_steps = [],
)

### タスク 2.10: パイプラインの定義

ステップを定義したら、**pipeline** という名前の変数でパイプラインを設定します。以前に定義したステップがパイプライン定義にどのように渡されるかに注目してください。

In [ ]:
 #define pipeline function
def get_pipeline(
    region,
    role = None,
    default_bucket = None,
    model_package_group_name = "ChurnModelPackageGroup",
    pipeline_name = "ChurnModelPipeline",
    base_prefix = None,
    custom_image_uri = None,
    sklearn_processor_version = None
    ):

    #configure pipeline instance
    pipeline = Pipeline(
        name = pipeline_name,
        parameters = [
            processing_instance_type,
            processing_instance_count,
            training_instance_type,
            input_data,
            batch_data,
        ],
        steps = [step_process, step_tuning, step_eval, step_cond],
        sagemaker_session = sagemaker_session
    )
    return pipeline


### タスク 2.11: パイプラインの作成

In [ ]:
 #create pipeline using function
pipeline = get_pipeline(
  region = region,
    role = role,
    default_bucket = default_bucket,
    model_package_group_name = model_package_group_name,
    pipeline_name = pipeline_name,
    custom_image_uri = clarify_image,
    sklearn_processor_version = sklearn_processor_version
)

### タスク 2.12: 正しい IAM ロールを使用するようにパイプラインを更新する

In [ ]:
#set-iam-role
pipeline.upsert(role_arn = role)

**注:** セルを実行した後に sagemaker.workflow の警告が表示された場合は、無視しても問題ありません。

### タスク 2.13: パイプラインを開始する

In [ ]:
#start-pipeline
RunPipeline = pipeline.start()

### タスク 2.14: パイプラインの説明

In [ ]:
#describe-pipeline
RunPipeline.describe()

このパイプラインの実行には約 35 分かかります。

パイプラインの実行中に、次のタスクに進んで Amazon SageMaker Studio コンソールでパイプラインを調べてください。

## パイプラインのモニタリングと承認

このタスクでは、Amazon SageMaker Studio コンソールを使用してパイプラインを調べます。

### タスク 2.15: SageMaker Studio でパイプラインをモニタする

1. **SagemakerStudioUrl** の値をこれらの指示の左にコピーしてください。

1. 新しいブラウザタブを開き、**SagemakerStudioUrl** の値をアドレスバーに貼り付けます。

1. **Enter** を押してください。

ブラウザに SageMaker Studio ページが表示されます。

**注:** JupyterLab ワークスペースタブと SageMaker Studio タブを並べて表示できます。これにより、パイプラインのステップを調べるときに方向を表示することができます。

1. SageMaker Studio のウェルカムポップアップウィンドウで、**Skip Tour for now** を選択します。

1. 左パネルのナビゲーションメニューで、**Pipelines** を選択します。

SageMaker Studio が **Pipelines** ページを開きます。

1. **ChurnModelSMPipeline** という名前のパイプラインを選択してください。

SageMaker Studio は **ChurnModelSMPipeline** ページを開き、**Executions** タブに実行の詳細とステータスを表示します。

1. **Name** 列の下にある実行のリンクを選択してください。

このページには、実行のすべてのステップを示す実行グラフが表示されます。各ステップが実行中か、実行待ちか、正常に完了したか、エラーが発生したかを確認できます。

グラフ内の任意のステップを選択して、ステップの詳細を表示することもできます。

1. グラフで **ChurnModelProcess** という名前のステップを選択 (ダブルクリック) します。**ChurnModelProcess** という名前の新しいペインが表示されます。

1. **ChurnModelProcess** ペインで、このパイプラインステップに関連するタブを確認してください。
    - **Overview** タブを選択してください。このタブには、ステップステータスに関する情報が含まれています。このタブには、パイプラインステップで生成されるさまざまなファイルとその配置場所も表示されます。パイプラインはすべての出力を SageMaker Studio のデフォルトバケットに配置します。
    - **Settings** タブを選択してください。このタブには、処理ステップで使用されるパラメーターとファイルに関する役立つ情報が含まれています。パラメーターセクションには、ジョブが使用するインスタンスタイプとイメージ、データセットの場所、コードの場所、生成されるさまざまな出力の宛先などの詳細が表示されます。ペインの一番下までスクロールして、ジョブに渡されたファイル入力を探します。
    - **Details** タブを選択してください。このタブには、このステップが関連付けられているパイプラインの詳細が表示されます。また、**MetaData** セクションにはステップタイプも表示されます。このタブには、ジョブが生成するログも表示されます。SageMaker Studio 内でログを使用できるようにしておくと、パイプラインステップが正常に実行されなかった場合の調査とトラブルシューティングを迅速に行うことができます。

### タスク 2.16: パイプラインステップの詳細を確認

次のステップでは、実行グラフから適切なステップを選択して、特定のパイプラインステップに関する情報を検索します。答えを見つけるのに助けが必要な場合、正解またはヒントがこのノートブックの最後に記載されています。

1. **ChurnHyperParameterTuning** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か
    - このステップで生成された **Training Job** は何か
1. **ChurnEvalBestModel** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か
    - 前のステップで特定した最上位モデルの評価に使用する Python スクリプトの名前は何か
    - このファイルはどこにあるか
    - このステップの結果はどこに書かれたか
1. **CheckAUCScoreChurnEvaluation** という名前のステップについて、次の詳細を確認してください。n    - このステップの **Step Type** は何か
    - **Evaluation outcome** は何か
1. **ChurnCreateModel** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か
    - このジョブはログを生成したか
1. **RegisterChurnModel-RegisterModel** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か
1. **ChurnTransform** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か
    - このジョブはログを生成したか
    - このステップではどのファイルが入力されたか
1. **ChurnModelConfigFile** という名前のステップについて、次の詳細を確認してください。
    - このジョブの実行に使用された処理インスタンスタイプはどれか
    - このステップの **Step Type** は何か
1. **ClarifyProcessingStep** という名前のステップについて、次の詳細を確認してください。
    - このステップで出力されたファイルは何か
    - 出力はどこに書かれたか

### 2.17: パイプライン内のモデルを承認する

パイプラインの実行が終了したら、パイプラインが作成したモデルを表示します。

1. SageMaker Studio の左側のナビゲーションパネルから **Models** を選択します。

1. **Models** ページで、**ChurnModelPackageGroup** リンクを選択します。

1. 表に表示されているバージョンリンクを選択します。

    SageMaker Studio には、選択したバージョンの概要ページが表示されます。

    - モデルのデプロイステータスが **Pending Approval** であることに注意してください。

    パイプラインに関するその他の詳細は、**Activity:** と **Details** のタブにあります。

1. モデルを承認してください。このプロセスは、モデルを承認する前に手動で確認するように設計されています。ただし、パイプライン内のモデル承認を自動化することは可能です。
    - **Overview** タブと **Deploy** タイルから選択してください。
    - **Approved** を選んでください。
    - **Save** を選んでください。.

### タスク 2.18: Amazon SageMaker Python SDK を使用してパイプラインステップを表示する

SageMaker Studio UI を使用してパイプラインの詳細を表示する以外に、Amazon SageMaker Python SDK コマンドを使用することもできます。たとえば、次の Boto3 コマンドはパイプラインステップのリストを返します。

In [ ]:
#list-steps
RunPipeline.list_steps()

## アーティファクトを確認

### タスク 2.19: Amazon SageMaker Clarify 分析を確認して、デフォルトの SageMaker S3 バケット内のパイプラインアーティファクトを探す

パイプラインによって生成されたアーティファクトは、デフォルトの SageMaker Amazon S3 バケットに保存されます。AWS マネジメントコンソールから Amazon S3 に移動すると、これらのアーティファクトを確認できます。代わりに、このラボでは JupyterLab ワークスペースの Amazon S3 ブラウザ拡張機能がインストールされています。これにより、JupyterLab ワークスペース内から Amazon S3 に保存されているオブジェクトに直接アクセスできます。

1. JupyterLab ワークスペースで、左側のパネルにある **Object Storage Browser** アイコンを選択します。

1. **sagemaker-** で始まる名前のバケットと AWS リージョンを選択してください (例:**sagemaker-us-west-2-123456789**)。

1. **clarify-output/bias/** サブフォルダーに移動します。

1. **report.ipynb** ノートブックファイルを開きます。

1. **Select Kernel** ウィンドウの **Select kernel for: "report.ipynb"** で **Python 3 (ipykernel)** を選択し、**Select** を選択してからを選択します。

1. Amazon SageMaker Clarify 分析のアウトプットを調べてください。

デフォルトの SageMaker S3 バケット内の他のアーティファクトはすべて調べることができます。

## (任意) パイプラインの系統を構築して確認する

モデルがどのように予測を行うかを説明し、モデルの潜在的な偏りを理解するのに役立つ SageMaker Clarify の使い方を学びました。また、SageMaker Clarify を使用して、モデル監査でしばしば必要となるモデル生成の手順を確認することもできます。このタスクでは、mlLineageHelper モジュールを利用して現在のパイプラインランの系統を構築します。ML リネージュヘルパーの詳細については、[MLLineageHelper](https://github.com/aws-samples/ml-lineage-helper) を参照してください。

Amazon SageMaker ML Lineage Tracking は、データ準備からモデル展開までの ML ワークフローのステップに関する情報を作成して保存します。追跡情報があれば、ワークフローステップを再現し、モデルとデータセットのリネージを追跡し、モデルガバナンスと監査基準を確立できます。

### タスク 2.21: セッションと変数の設定

In [ ]:
#set-variables
fs_query = feature_group.athena_query()
fs_table = fs_query.table_name
query_string = 'SELECT * FROM "'+fs_table+'"'

### タスク 2.22: モデルの系統を構築するために使用される値を表示

設定の構成には以下が含まれます。
- **query_string:** これは mlLineageHelper モジュールに渡される SageMaker フィーチャストアクエリです。
- **model_ref:** これは評価中のモデルの名前です。
- **processing_job:** これはモデルを生成した処理ジョブの名前です。

In [ ]:
#print-values
print ('query_string:',query_string)

model_ref = sagemaker_client.list_models(SortBy = 'CreationTime', SortOrder = 'Descending')['Models'][0]['ModelName']
print ('model_ref:',model_ref)

processing_job = sagemaker_client.list_processing_jobs(SortBy = 'CreationTime', SortOrder = 'Descending', NameContains = 'ChurnModelProcess')['ProcessingJobSummaries'][0]['ProcessingJobName']
print ('processing_job:',processing_job)

processing_job_description = sagemaker_client.describe_processing_job(
    ProcessingJobName = processing_job
    )

### タスク 2.23: 処理ジョブの説明

In [ ]:
#describe-processing-job
processing_job_description

### タスク 2.24: モデルの作成に使用したトレーニングジョブの名前を表示する

In [ ]:
#print-training-job
training_job_name  =  sagemaker_client.list_training_jobs(SortBy = 'CreationTime', SortOrder = 'Descending')['TrainingJobSummaries'][0]['TrainingJobName']
print (training_job_name)

### Task 2.25: Build the lineage for the model

次のエラーが表示された場合は、セルを再実行してください。
- **ClientError: An error occurred (ThrottlingException) when calling the UpdateArtifact operation (reached max retries: 4): Rate exceeded**

In [ ]:
#build-lineage
ml_lineage = MLLineageHelper()
lineage = ml_lineage.create_ml_lineage(training_job_name, model_name = model_ref,
                                       query = query_string, sagemaker_processing_job_description = processing_job_description,
                                       feature_group_names = [feature_group_name])

### タスク 2.26: 系統を現在のトライアルと機能グループのみに制限する

パイプラインは複数回実行できます。最新のトレーニングジョブ実行の詳細を確実に取得するには、現在のトライアルの名前とトライアルが使用する機能グループを使用して系統呼び出しをフィルタリングします。

このセルを実行すると、モデルの作成に使用されたステップ、ステップの実行順序、およびパイプライン内の他のジョブに寄与したジョブが表として表示されます。これと同じ情報が **lineage_FS.csv** という名前のファイルにも書き込まれます。このファイルをダウンロードして出力を保存し、監査人など他のチームメンバーと共有できます。

In [ ]:
#limit-lineage
trial_name = RunPipeline.describe()['PipelineExperimentConfig']['TrialName']
pat = str(trial_name)+'|'+'fg-FG'
df1 = lineage[lineage.apply(lambda x: any(x.str.contains(pat)),axis = 1)]
pd.set_option('display.max_colwidth', 120)
df1.to_csv('lineage_FS.csv') 
df1

### タスク 2.27: モデルの系統のビジュアライゼーションを生成する

In [ ]:
#visualize-lineage
plt.figure(3, figsize = (20, 14))
graph = nx.DiGraph()
graph.add_edges_from([(each[0], each[2]) for each in df1.values])
fig, ax = plt.subplots()
nx.draw_networkx(
    graph,
    node_size = 300,
    node_color = "orange",
    alpha = 0.65,
    font_size = 8,
    pos = nx.spring_layout(graph)
)
ax.set_facecolor('deepskyblue')
ax.axis('off')
fig.set_facecolor('deepskyblue')
plt.show()

### タスク 2.28: パイプラインの削除

パイプラインを削除するには、次のセルを実行してください。

In [ ]:
#delete-pipeline
response = sagemaker_client.delete_pipeline(PipelineName = 'ChurnModelSMPipeline')
print (response)

### まとめ

お疲れ様でした。Amazon SageMaker Pipelines を使用してモデルの作成と登録を自動化しました。パイプラインの各ステップを掘り下げて、関連するパラメーター、ファイル、ログを特定する方法を学びました。パイプラインがモデルの生成に使用したアセットを特定する方法、モデルレジストリでモデルを検索する方法、パイプラインが生成できる説明可能性レポートとバイアスレポートを見つけて表示する方法を理解しています。

### クリーンアップ

このノートが完成しました。ラボの次のパートに進むには、次の操作を行います。

- このノートブックファイルを閉じます。
- ラボセッションに戻って **まとめ** に進んでください。

### タスク 2.16 のヒントと回答 : パイプラインステップの詳細を確認
一般的なヒント:**Step type** は **Details** タブの **Metadata** セクションにあります。

1. **ChurnHyperParameterTuning** という名前のステップについて、次の詳細を確認してください。
    - このステップの **Step Type** は何か?</br>
    **ヒント:** この情報は **Metadata** セクションの **Details** タブにあります。</br>
    **解答:** チューニング</br>
    - このステップで生成された **Training Job** は何か </br>
    **ヒント:** この情報は **Overview** タブにあります。</br>
    **解答:** モデル名は生成され、学生ごとに異なります。名前は次の例のようになるはずです: 056vhzs2vkxc-ChurnHy-TCAtUr16oV-001-17d5bd01
1. **ChurnEvalBestModel** ステップでは、次の詳細を確認してください。
    - このステップの **Step Type** は何か
    **解答:** 処理中</br>
    - 前のステップで特定した最上位モデルの評価に使用する Python スクリプトの名前は何か</br>
    **ヒント:** この情報は **Files** セクションの **Settings** タブにあります。</br>
    **解答:** evaluate.py</br>
    - このファイルはどこにあるか</br>
    **ヒント:** この情報は **Files** セクションの **Settings** タブにあります。</br>
    **解答:** ファイルは S3 バケットにあります。パスは次の例のようになります。s3://sagemaker-us-west-2-1234567890/input/code/evaluate.py</br>
    - このステップの結果はどこに書かれてるか</br>
    **ヒント:** この情報は **Settings** タブの **Parameters** セクションにあり、次に **evaluation** セクションにあります。</br></br>
    **解答:** 評価の結果は S3 バケットに書き込まれました。ファイルへのパスは次の例のようになっているはずです。s3://sagemaker-us-west-2-1234567890/output/evaluation</br>
1. **CheckAUCScoreChurnEvaluation** ステップでは、次の詳細を確認してください。
    - このステップの **Step Type** は何か</br>
    **解答:** 状態</br>
    - **Evaluation outcome** って何か?</br>
    **ヒント:** この情報は **Metadata** セクションの **Details** タブにあります。</br>
    **解答:** True
1. **ChurnCreateModel** ステップでは、次の詳細を確認してください。
    - このステップの **Step Type** は何か</br>
    **解答:** モデル</br>
    - このジョブはログを生成したか</br>
    **解答:** いいえ
1. **RegisterChurnModel-RegisterModel** ステップでは、次の詳細を確認してください。
    - このステップの **Step Type** は何か</br>
    **解答:** RegisterModel</br>
1. **ChurnTransform** ステップでは、次の詳細を確認してください。
    - このステップの **Step Type** は何か</br>
    **解答:** Transform</br>
    - このジョブはログを生成したか</br>
    **解答:** はい</br>
    - このステップではどのファイルが入力されたか</br>
    **ヒント:** この情報は **Files** セクションの **Settings** タブにあります。ファイル名を見つけるには、ウィンドウの一番下までスクロールする必要がある場合があります。</br>
    **解答:** model.tar.gz, sagemaker-xgboost:1.5-1-cpu-py3, batch.csv
1. **ChurnModelConfigFile** ステップでは、次の詳細を確認してください。
    - このジョブの実行に使用された処理インスタンスタイプはどれか</br>
    **ヒント:** この情報は **Settings** タブにあります。</br>
    **解答:** ml.m5.xlarge
    - Wこのステップの **Step Type** は何か</br>
    **解答:** 処理中
1. **ClarifyProcessingStep** については、以下の詳細を確認する
    - このステップで出力されたファイルは何か
    **ヒント:** この情報は **Files** セクションの **Settings** タブにあります。</br>
    **解答:** 出力はバイアスデータでした。
    - 出力はどこに書かれたか
    **解答:** 出力は S3 バケットに書き込まれました。パスは次の例のようになっているはずです: s3://sagemaker-us-west-2-1234567890/clarify-output/bias